# Rosaloid Demo

This notebook demonstrates the Rosaloid pipeline for protein optimization using **ESM-2 embeddings** and **NGBoost surrogate modeling** with **Bayesian Optimization**.

The core logic has been refactored into the `src` package. This notebook serves as a high-level driver.

In [ ]:
%load_ext autoreload
%autoreload 2
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt

# Add project root to path
sys.path.append(os.path.abspath(".."))

from src.bo_loop import run_bo_loop
from src.utils import DMS_ID_TO_FILE

## Configuration

Select the protein target (DMS_id) and experiment parameters.

In [ ]:
DMS_ID = "GFP_AEQVI_Sarkisyan_2016"
OUTPUT_DIR = "../results"
ROUNDS = 5
BATCH_SIZE = 24
DEVICE = "cuda" # or "cpu"

## Run Optimization Loop

This single command runs the full BO pipeline:
1. Loads data
2. Computes/Loads ESM-2 embeddings
3. Trains NGBoost surrogate
4. Proposes diverse candidates via Expected Improvement
5. Simulates feedback and repeats

In [ ]:
metrics_df = run_bo_loop(
    dms_id=DMS_ID,
    output_dir=OUTPUT_DIR,
    rounds=ROUNDS,
    batch_size=BATCH_SIZE,
    device=DEVICE
)

## Analyze Results

In [ ]:
# Display metrics table
display(metrics_df)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 1. Best so far
axes[0].plot(metrics_df["round"], metrics_df["best_so_far"], marker='o')
axes[0].set_title("Best Fitness Found")
axes[0].set_xlabel("Round")
axes[0].set_ylabel("Fitness (Max)")

# 2. Diversity
axes[1].plot(metrics_df["round"], metrics_df["diversity_mean_hamming"], marker='o', color='orange')
axes[1].set_title("Batch Diversity (Hamming)")
axes[1].set_xlabel("Round")

# 3. Hit@96
axes[2].plot(metrics_df["round"], metrics_df["Hit@96"], marker='o', color='green')
axes[2].set_title("Hit@96 (Top True Found)")
axes[2].set_xlabel("Round")

plt.tight_layout()
plt.show()